In [33]:
print("helloworld")

helloworld


In [34]:
import sys
!{sys.executable} -m pip install -U pip setuptools wheel
!{sys.executable} -m pip uninstall -y fasttext
!{sys.executable} -m pip install fasttext-wheel
!{sys.executable} -m pip install nltk

In [35]:
import os, urllib.request, gzip, shutil

NB_GZ = "numberbatch-en-19.08.txt.gz"
NB_TXT = "numberbatch-en-19.08.txt"
NB_URL = "https://conceptnet.s3.amazonaws.com/downloads/2019/numberbatch/numberbatch-en-19.08.txt.gz"

if not os.path.exists(NB_GZ) and not os.path.exists(NB_TXT):
    print("Downloading Numberbatch...")
    urllib.request.urlretrieve(NB_URL, NB_GZ)
    print("Downloaded:", NB_GZ)

if os.path.exists(NB_GZ) and not os.path.exists(NB_TXT):
    print("Extracting Numberbatch...")
    with gzip.open(NB_GZ, "rb") as f_in, open(NB_TXT, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
    print("Extracted:", NB_TXT)


In [44]:
import os
import re
import numpy as np
import fasttext
import nltk
from nltk.corpus import stopwords

In [ ]:


MODEL_PATH = r"C:\Users\kyabr\PersonalProjects\HackaCookers\models\cc.en.300.bin"

try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))

# Make sure the file exists
if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found:\n{MODEL_PATH}")

In [ ]:


print("Loading fastText model (this can take a while and may use lots of RAM)...")
model = fasttext.load_model(MODEL_PATH)  # <-- no mmap on your build
print("Model loaded.")

In [ ]:


def tokenize(text: str) -> list[str]: #convert caption to list of strings
    text = text.lower() #put everything lowercase
    text = re.sub(r"#", " ", text) #remove hashtags"
    text = re.sub(r"[^a-z\s]", " ", text) #get rid of everything except lowercase letters and spaces

    tokens = text.split() #split string into words
    cleaned = []

    for t in tokens:
        if len(t) <= 2: #remove short words
            continue
        if t.endswith("s") and not t.endswith("ss"):
            t = t[:-1]

        if t in STOPWORDS: #get rid of stopwords
            continue
        cleaned.append(t)

    return cleaned


def cosine(a: np.ndarray, b: np.ndarray) -> float: #method of measuring semantic simularity
    denom = np.linalg.norm(a) * np.linalg.norm(b) #vector magnitude for normalization
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom) #cosine similarity

def relatedness_score(word: str, caption: str, tau: float = 0.35, k: float = 12.0, high: float = 0.45, bonus: float = 0.05) -> float:
    tokens = tokenize(caption) 
    if not tokens:
        return 0.0

    q = model.get_word_vector(word)
    sims = np.array([cosine(q, model.get_word_vector(tok)) for tok in tokens], dtype=np.float32)

    best = float(sims.max())
    count = int(np.sum(sims >= high))

 
    base = 1.0 / (1.0 + np.exp(-k * (best - tau)))

    # small bonus for multiple strong hits
    score = base + bonus * max(0, count - 1)
    return float(min(1.0, score))


# Notebook test run:
tests = [
    ("girl", "Fortnite update just dropped and it's insane"),
    ("girl", "Females are crazy"),
    ("girl", "female minds man"),
    ("pie", "let's cook up something tonight"),
    ("pie", "it was a huge disaster"),
    ("puppy", "I want a dog"),
    ("puppy", "women are cool"),
    ("oil", "environmentalism is very important for us to work with"),
    ("oil", "I'm a blue collar guy"),
    ("oil", "I wonder what it's for dinner"),
    ("oil", "I am a strong man"),
    ("oil", "tar is good for you"),
    ("oil", "I'm a chemist in my day job"),
    ("republican", "Today I'm thrilled to announce my plan to lower healthcare prices for all Americans and truly make healthcare affordable again. We're doing things that nobody's ever been able to do. We're calling it the Great Healthcare Plan instead of putting the needs of big corporations and special interest first. Our plan finally puts you first and puts more money in your pocket. The government is going to pay the money directly to you. It goes to you and then you take the money and buy your own healthcare. Nobody's ever heard of that before and that's the way it is. The big insurance companies lose and the people of our country win. This proposal locks in the massive discounts on prescription drugs that my administration is achieving through our most favored nation drug pricing agreement. Now when you hear about that for 40 years I've been trying to do it. But they never were able to do it. No other president was able to do it. I got every other country to approve it by the use of tariffs and other things. They all approved it. Nobody else got it. No other president got it. And for the most part that didn't even try because they felt it was impossible. It'll bring down drug prices 80, 90 percent in some cases just numbers that nobody's ever heard of before. Your prescription drugs will come way, way down. And under this policy the prices of many drugs will be slashed by 300, 400 and even 500 percent starting this month at thetrupprx.gov. So instead of Americans paying the highest drug prices in the world which we have for decades we will now be paying the lowest cost paid by any of the nations. So any of the nations that's paying the lowest cost that's what we're going to pay. And the American people will get the savings. So I have to reiterate, the lowest price in the world is what you're going to pay. Before you were paying the highest price in the world by far and the politicians did nothing about it. So I'm asking Congress to complete the work that we've started. Next my plan would reduce your insurance premiums by stopping government payoffs to big insurance companies and sending that money directly to the people. Obamacare was designed to make insurance companies rich. I call it the unaffordable care act with billions of dollars in taxpayer subsidies that help their stock prices skyrocket over 1,700 percent. As you paid more money for healthcare every single year, more and more, the premiums went higher and higher. I want to end this flagrant scam and put extra money straight into the healthcare savings account in your name and you go out and buy your own healthcare and you'll make a great deal. You get better healthcare for less money. That way you can choose the care that is right for your family. To further reduce insurance premiums, my plan ends the giant kickbacks to insurance brokers and corporate middlemen that only drive up the costs and that's what they're intended to drive up the cost. But we're driving down the costs. And it fully funds a long, neglected part of the law known as the cost sharing reduction program. This measure alone should cut premiums on the most popular Obamacare plans. It's hard to believe there are any because it's a hated program. It's unaffordable. But it's going to cut them by an average of 10 to 15 percent. Next, the great healthcare plan. That's the name. It's called the great healthcare because it's great healthcare at a lower price. It's unprecedented accountability and transparency from insurance companies and all healthcare providers so that special interests can no longer profiteer at your expense. As the saying goes, sunlight is the best disinfectant. That is why my plan orders all insurance companies to publish rate and coverage comparisons in very plain English. It requires insurers to publish detailed information about how much of your money. They're going to be paying out in claims versus how much they're taken in in profits. In other words, you will be able to watch the scam. It forces them to release detailed data on how many claims are being denied and whether those denials are eventually overturned on appeal. And most importantly, it will require any hospital or insurance who accepts a Medicare or Medicaid to prominently post all prices at their place of business so that you are never surprised and you can easily shop for a better deal or better care. And you're going to end up doing both. You're going to get a better deal and better care. We will have maximum price transparency and costs will come down incredibly. I'm calling on Congress to pass this framework into law without delay. I'm going to have to do it right now so that we can get immediate relief to the American people, the people I love. Thank you very much.")]

for w, c in tests:
    score = relatedness_score(w, c)
    print(f"{w!r} vs {c!r} → score={score:.3f}")


Loading fastText model (this can take a while and may use lots of RAM)...
Model loaded.
'girl' vs "Fortnite update just dropped and it's insane" → score=0.091
'girl' vs 'Females are crazy' → score=0.828
'girl' vs 'female minds man' → score=0.970
'pie' vs "let's cook up something tonight" → score=0.353
'pie' vs 'it was a huge disaster' → score=0.113
'puppy' vs 'I want a dog' → score=0.995
'puppy' vs 'women are cool' → score=0.039
'oil' vs 'environmentalism is very important for us to work with' → score=0.174
'oil' vs "I'm a blue collar guy" → score=0.074
'oil' vs 'I wonder what is for dinner' → score=0.043
'oil' vs 'I am a strong man' → score=0.062
'oil' vs 'tar is good for you' → score=0.684
'oil' vs "I'm a chemist in my day job" → score=0.143
'republican' vs 'But the weight lifting, but no, the girl gets up. And you see, I want to be more, but I have somebody watch it. I want to be more of you, sir. I want to really, yeah. But she gets it. Drop the thing, walks off the stage crying, h